In [ ]:
!pip install gradio

In [ ]:
!pip install pyngrok

In [12]:
%%writefile app1.ipynb
import streamlit as st
import tensorflow as tf
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

# Load model
model = tf.keras.models.load_model("/content/drive/MyDrive/age_model.h5")

st.title("Age Prediction using CNN")
st.write("Upload a face image")

# Upload button
uploaded_file = st.file_uploader("Choose an image...", type=["jpg","png","jpeg"])

if uploaded_file is not None:
    image = Image.open(uploaded_file)
    st.image(image, caption="Uploaded Image", use_column_width=True)

    # Preprocess
    img = np.array(image)
    img = cv2.resize(img, (128,128))
    img_array = np.expand_dims(img.astype(np.float32)/255.0, axis=0)

    # Predict
    predicted_age = model.predict(img_array)[0][0]

    st.write(f"Predicted Age: {predicted_age:.1f}")

Writing app1.ipynb


In [13]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Load model
model = tf.keras.models.load_model('/content/drive/MyDrive/age_model.h5')

# Get last conv layer
last_conv_layer = model.get_layer("last_conv")

# Model to get conv outputs
last_conv_model = tf.keras.Model(
    inputs=model.inputs,
    outputs=last_conv_layer.output
)

# Build classifier model
classifier_input = tf.keras.Input(shape=last_conv_layer.output.shape[1:])
x = classifier_input

start_collecting = False
for layer in model.layers:
    if layer.name == "last_conv":
        start_collecting = True
        continue
    if start_collecting:
        x = layer(x)

classifier_model = tf.keras.Model(classifier_input, x)

# Grad-CAM function
def make_gradcam_heatmap(img_array):
    img_tensor = tf.convert_to_tensor(img_array, dtype=tf.float32)

    with tf.GradientTape() as tape:
        conv_outputs = last_conv_model(img_tensor)
        tape.watch(conv_outputs)

        predictions = classifier_model(conv_outputs)
        loss = predictions[:, 0]

    grads = tape.gradient(loss, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))
    conv_outputs = conv_outputs[0]

    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    heatmap = tf.maximum(heatmap, 0)
    heatmap /= tf.reduce_max(heatmap)

    return heatmap.numpy()

In [14]:
import gradio as gr
import cv2
import numpy as np

def predict(image):
    img = cv2.resize(image, (128,128))
    img_array = np.expand_dims(img.astype(np.float32)/255.0, axis=0)

    predicted_age = model.predict(img_array)[0][0]

    heatmap = make_gradcam_heatmap(img_array)
    heatmap = cv2.resize(heatmap, (128,128))
    heatmap_uint8 = np.uint8(255 * heatmap)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)

    # Assume 'overlay' and 'predicted_age' are computed here

    overlay = heatmap_color * 0.4 + img


    output_image = cv2.cvtColor(np.uint8(overlay), cv2.COLOR_BGR2RGB)

    return output_image, f"Predicted Age: {predicted_age:.1f}"

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="numpy"),
    outputs=[
        gr.Image(label="Grad-CAM Output"),
        gr.Text(label="Prediction")
    ]
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c21b658bf650583d87.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
